# Week 3: Transfer Learning — mT5 Fine-Tuned for Dholuo

**Trizah — mT5-small, English/Kiswahili → Dholuo**

## Objective
Fine-tune `google/mt5-small` to translate Public Service Announcement (PSA) content
from **English** and **Kiswahili** into **Dholuo**.

## Language adaptation approach
mT5 has no per-language embedding table (unlike NLLB), so there is no language token to
add or resize. Since mT5 was pretrained on mC4, which does **not** include Dholuo, this is
treated as **partial support**: the model must learn the language largely from this
fine-tuning data, guided by a **consistent text-prefix scheme** rather than a language
embedding. Every source sentence is prefixed with an explicit natural-language instruction
(`"translate English to Dholuo: "` / `"tafsiri Kiswahili kwa Dholuo: "`) so the model has a
constant, learnable signal for which task it is performing — this is the standard mT5/T5
mechanism for multi-task conditioning in the absence of dedicated language tokens.

This notebook picks up directly from `dholuo_somali_preprocessing_eda.ipynb`, which produced
the cleaned dataset used below.

## Step 1: Install Required Libraries

In [1]:
!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate scikit-learn protobuf

## Step 2: Import Libraries

In [2]:
import pandas as pd
import numpy as np
import torch
import gc

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.13.0+cu130
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [3]:
# !git clone https://github.com/SelmahT/psa-dholuo-mt.git /home/jovyan/mt5-project
!ls -la /home/jovyan/mt5-project

total 72
drwxr-sr-x 10 jovyan users  4096 Aug  2 14:09 .
drwsrws---  1 jovyan users  4096 Aug  2 14:03 ..
drwxr-sr-x  3 jovyan users  4096 Aug  2 14:09 checkpoints
drwxr-sr-x  7 jovyan users  4096 Aug  2 14:03 data
drwxr-sr-x  2 jovyan users  4096 Aug  2 14:03 docs
drwxr-sr-x  8 jovyan users  4096 Aug  2 14:03 .git
-rw-r--r--  1 jovyan users   446 Aug  2 14:03 .gitignore
drwxr-sr-x  3 jovyan users  4096 Aug  2 14:09 models
drwxr-sr-x  4 jovyan users  4096 Aug  2 14:03 notebooks
-rw-r--r--  1 jovyan users 18178 Aug  2 14:03 README.md
drwxr-sr-x  2 jovyan users  4096 Aug  2 14:03 reports
-rw-r--r--  1 jovyan users   132 Aug  2 14:03 requirements.txt
drwxr-sr-x  2 jovyan users  4096 Aug  2 14:03 src


## Step 3: Mount Google Drive

Required so checkpoints and the final model survive a Colab runtime disconnect
(as happened during Week 2). Point `DRIVE_PROJECT_DIR` at wherever you keep the
project in your own Drive.

In [3]:
DRIVE_PROJECT_DIR = "/home/jovyan/mt5-project"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints/mt5-dholuo"
MODEL_OUT_DIR = f"{DRIVE_PROJECT_DIR}/models/mt5-dholuo-final"
DATA_PATH = f"{DRIVE_PROJECT_DIR}/data/processed/psa_dataset_dholuo_somali_cleaned.csv"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODEL_OUT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)
print("Final model will be saved to:", MODEL_OUT_DIR)


Checkpoints will be saved to: /home/jovyan/mt5-project/checkpoints/mt5-dholuo
Final model will be saved to: /home/jovyan/mt5-project/models/mt5-dholuo-final


## Step 4: Load the Cleaned Dataset

Loads the output of `dholuo_somali_preprocessing_eda.ipynb`
(`psa_dataset_dholuo_somali_cleaned.csv`, 16,029 rows). Only the columns needed
for the Dholuo track are kept — `Somali` is dropped (that's Patricia's target
language, not part of this notebook).

In [4]:
# DATA_PATH already set above:
# DATA_PATH = f"{DRIVE_PROJECT_DIR}/data/processed/psa_dataset_dholuo_somali_cleaned.csv"

df = pd.read_csv(DATA_PATH)
df = df[["PSA_Id", "Domain", "English", "Kiswahili", "Dholuo", "Class", "Source"]].copy()

print("Dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
df.head()


Dataset shape: (16029, 7)

Missing values:
 PSA_Id       0
Domain       0
English      0
Kiswahili    0
Dholuo       0
Class        0
Source       0
dtype: int64


,PSA_Id,Domain,English,Kiswahili,Dholuo,Class,Source
0,1,Education,Comprehensive COVID-19 health and safety proto...,Itifaki kamili za afya na usalama za COVID-19 ...,Chenro mag thieth: Chenro mag thieth kod ritru...,PSA,original_baseline_dataset
1,2,Education,Digital learning platform providing free educa...,Jukwaa la kujifunza kidijitali linalotoa maudh...,Kenya Education Cloud: Ohinga mar somo mar dij...,PSA,original_baseline_dataset
2,3,Education,KUCCPS portal will open in March 2025 for univ...,Lango la KUCCPS litafunguliwa Machi 2025 kwa n...,KUCCPS Portal: KUCCPS portal biro yawore e dwe...,PSA,original_baseline_dataset
3,4,Education,Target to increase school feeding beneficiarie...,Lengo ni kuongeza wanufaika wa chakula shuleni...,Medo kwan mar pidho nyithindo e skunde: Dwaro ...,PSA,original_baseline_dataset
4,5,Education,Launch of inclusive education programs with as...,Uzinduzi wa programu za elimu jumuishi zenye t...,Somo mar Dwaro Manyien: Chako chenro mag somo ...,PSA,original_baseline_dataset


## Step 5: Build Translation Pairs (with Dholuo prefix scheme)

Each row becomes **two** training examples: English→Dholuo and Kiswahili→Dholuo.
The explicit prefix is the language-adaptation workaround for mT5 (see intro cell).

In [5]:
ENG_PREFIX = "translate English to Dholuo: "
SWH_PREFIX = "tafsiri Kiswahili kwa Dholuo: "

translation_pairs = []

for _, row in df.iterrows():
    translation_pairs.append({
        "source_text": ENG_PREFIX + str(row["English"]),
        "target_text": str(row["Dholuo"]),
        "source_lang": "eng",
        "target_lang": "luo",
        "domain": row["Domain"],
    })
    translation_pairs.append({
        "source_text": SWH_PREFIX + str(row["Kiswahili"]),
        "target_text": str(row["Dholuo"]),
        "source_lang": "swh",
        "target_lang": "luo",
        "domain": row["Domain"],
    })

translation_df = pd.DataFrame(translation_pairs)
translation_df["translation_direction"] = translation_df["source_lang"].map({
    "eng": "eng-luo",
    "swh": "swh-luo",
})

print("Translation pairs shape:", translation_df.shape)
translation_df.head()


Translation pairs shape: (32058, 6)


,source_text,target_text,source_lang,target_lang,domain,translation_direction
0,translate English to Dholuo: Comprehensive COV...,Chenro mag thieth: Chenro mag thieth kod ritru...,eng,luo,Education,eng-luo
1,tafsiri Kiswahili kwa Dholuo: Itifaki kamili z...,Chenro mag thieth: Chenro mag thieth kod ritru...,swh,luo,Education,swh-luo
2,translate English to Dholuo: Digital learning ...,Kenya Education Cloud: Ohinga mar somo mar dij...,eng,luo,Education,eng-luo
3,tafsiri Kiswahili kwa Dholuo: Jukwaa la kujifu...,Kenya Education Cloud: Ohinga mar somo mar dij...,swh,luo,Education,swh-luo
4,translate English to Dholuo: KUCCPS portal wil...,KUCCPS Portal: KUCCPS portal biro yawore e dwe...,eng,luo,Education,eng-luo


## Step 6: Quality Checks

Same checks the team applied for Ekegusii: missing values, empty strings, exact
duplicate pairs.

In [6]:
print("Missing values:\n", translation_df.isnull().sum())

empty_src = (translation_df["source_text"].str.strip() == "").sum()
empty_tgt = (translation_df["target_text"].str.strip() == "").sum()
print(f"\nEmpty source rows: {empty_src}")
print(f"Empty target rows: {empty_tgt}")

dupes = translation_df.duplicated(subset=["source_text", "target_text"]).sum()
print(f"Exact duplicate pairs: {dupes}")

translation_df = translation_df.drop_duplicates(subset=["source_text", "target_text"]).reset_index(drop=True)
print(f"\nFinal translation pair count after dedup: {len(translation_df)}")


Missing values:
 source_text              0
target_text              0
source_lang              0
target_lang              0
domain                   0
translation_direction    0
dtype: int64

Empty source rows: 0
Empty target rows: 0
Exact duplicate pairs: 0

Final translation pair count after dedup: 32058


## Step 7: Stratified Train / Validation / Test Split

80/10/10 split, stratified by `source_lang` so English and Kiswahili stay
proportionally represented in every split (same approach the team used for
Ekegusii).

In [7]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    translation_df,
    test_size=0.20,
    random_state=42,
    stratify=translation_df["source_lang"],
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["source_lang"],
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
validation_dataset = Dataset.from_pandas(validation_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print("Training examples  :", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("Test examples       :", len(test_dataset))

from collections import Counter
print("\nTrain source_lang distribution:", Counter(train_dataset["source_lang"]))
print("Val source_lang distribution  :", Counter(validation_dataset["source_lang"]))
print("Test source_lang distribution :", Counter(test_dataset["source_lang"]))


Training examples  : 25646
Validation examples: 3206
Test examples       : 3206

Train source_lang distribution: Counter({'swh': 12823, 'eng': 12823})
Val source_lang distribution  : Counter({'eng': 1603, 'swh': 1603})
Test source_lang distribution : Counter({'eng': 1603, 'swh': 1603})


## Step 8: Load Tokenizer and Model

In [8]:
import glob

checkpoints = sorted(glob.glob(f"{CHECKPOINT_DIR}/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
LATEST_CHECKPOINT = checkpoints[-1]
print(f"Using checkpoint: {LATEST_CHECKPOINT}")

MT5_MODEL = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(MT5_MODEL)
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

model = AutoModelForSeq2SeqLM.from_pretrained(LATEST_CHECKPOINT)
if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id
print(f"Model parameters: {model.num_parameters():,}")

Using checkpoint: /home/jovyan/mt5-project/checkpoints/mt5-dholuo/checkpoint-8020


Vocabulary size: 250,100


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 5126.72it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model parameters: 300,176,768


## Step 9: Tokenize Datasets

mT5 uses SentencePiece with no dedicated language tokens, so tokenization is
plain text-in/text-out — the prefix added in Step 5 is what carries the task
signal.

In [9]:
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 256

def tokenize_function(example):
    model_inputs = tokenizer(
        example["source_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        example["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing training set...")
tokenized_train = train_dataset.map(tokenize_function, remove_columns=train_dataset.column_names)
print("Tokenizing validation set...")
tokenized_val = validation_dataset.map(tokenize_function, remove_columns=validation_dataset.column_names)
print("Tokenizing test set...")
tokenized_test = test_dataset.map(tokenize_function, remove_columns=test_dataset.column_names)

print("\nSample tokenization:")
sample = train_dataset[0]
tok_sample = tokenize_function(sample)
print("Source:", sample["source_text"][:80], "...")
print("Source tokens:", len(tok_sample["input_ids"]))
print("Target tokens:", len(tok_sample["labels"]))


Tokenizing training set...


Map: 100%|██████████| 25646/25646 [00:10<00:00, 2519.10 examples/s]


Tokenizing validation set...


Map: 100%|██████████| 3206/3206 [00:01<00:00, 2400.58 examples/s]


Tokenizing test set...


Map: 100%|██████████| 3206/3206 [00:01<00:00, 2421.69 examples/s]


Sample tokenization:
Source: tafsiri Kiswahili kwa Dholuo: Mpango huu umeongozwa na WHO katika kanda ya Afrik ...
Source tokens: 54
Target tokens: 45


## Step 10: Data Collator

In [10]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=tokenizer.pad_token_id,
)
print("Data collator ready.")


Data collator ready.


## Step 11: Evaluation Metrics

BLEU, SacreBLEU, and chrF — matching the metrics used across the whole team's
models for a consistent, comparable Week 3 report.

In [11]:
bleu_metric = evaluate.load("bleu")
sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)  # <-- new line
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels_bleu = [[l.strip()] for l in decoded_labels]

    try:
        bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels_bleu)["bleu"]
    except Exception as e:
        print("BLEU failed:", e); bleu = 0.0

    try:
        sacrebleu = sacrebleu_metric.compute(predictions=decoded_preds, references=decoded_labels_bleu)["score"]
    except Exception as e:
        print("SacreBLEU failed:", e); sacrebleu = 0.0

    try:
        chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels_bleu)["score"]
    except Exception as e:
        print("chrF failed:", e); chrf = 0.0

    return {"bleu": bleu, "sacrebleu": sacrebleu, "chrf": chrf}

print("Metrics loaded: BLEU, SacreBLEU, chrF")


Metrics loaded: BLEU, SacreBLEU, chrF


## Step 12: Training Arguments

Per the Week 3 requirements: **10 epochs**, `save_strategy="epoch"` (automatic
checkpointing, no manual saves needed), checkpoints written to the mounted
Drive so nothing is lost on a runtime disconnect.

In [12]:
training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,
    weight_decay=0.01,
    num_train_epochs=10,
    predict_with_generate=False,
    generation_max_length=256,
    generation_num_beams=1,
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    report_to="none",
    save_total_limit=3,
    warmup_steps=100,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=None,
)

print("Trainer ready.")
print(f"Training examples  : {len(tokenized_train):,}")
print(f"Validation examples: {len(tokenized_val):,}")
print(f"Epochs: 10 | Batch size: 8 | Checkpoints: {CHECKPOINT_DIR}")


Trainer ready.
Training examples  : 25,646
Validation examples: 3,206
Epochs: 10 | Batch size: 8 | Checkpoints: /home/jovyan/mt5-project/checkpoints/mt5-dholuo


## Step 13: Train

In [14]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()

train_result = trainer.train()

print("Final training loss:", train_result.training_loss)
print("Training time (min):", train_result.metrics.get("train_runtime", 0) / 60)


Epoch,Training Loss,Validation Loss
1,8.827051,7.318746
2,3.747977,3.193068
3,3.245024,2.886844
4,2.789641,2.348637
5,2.109914,1.776483
6,1.877691,1.588331
7,1.771449,1.481988
8,1.670108,1.418644
9,1.664948,1.381941
10,1.620242,1.369326


[W802 14:49:20.322686492 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 7876902912 bytes (free: 4615634944, total: 85093777408).
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Final training loss: 4.304659074797595
Training time (min): 18.201745000000003


## Step 14: Per-Epoch Results Table

Pulls the epoch-by-epoch training/validation loss and metrics straight from
the trainer's log history — this is the table for **Report Section 3
(Results)**.

In [16]:
log_history = trainer.state.log_history

epoch_rows = []
for entry in log_history:
    if "eval_loss" in entry:
        epoch_rows.append({
            "Epoch": entry.get("epoch"),
            "Training Loss": None,  # filled in below from the preceding 'loss' entry
            "Validation Loss": entry.get("eval_loss"),
            "BLEU": entry.get("eval_bleu"),
            "SacreBLEU": entry.get("eval_sacrebleu"),
            "chrF": entry.get("eval_chrf"),
        })

train_losses = [e["loss"] for e in log_history if "loss" in e and "eval_loss" not in e]
for i, row in enumerate(epoch_rows):
    if i < len(train_losses):
        row["Training Loss"] = train_losses[min(i, len(train_losses)-1)]

results_df = pd.DataFrame(epoch_rows)
print(results_df.to_string(index=False))

best_epoch_idx = results_df["Validation Loss"].idxmin()  # lower loss = better, so idxmin not idxmax
print(f"\nBest epoch by Validation Loss: epoch {results_df.loc[best_epoch_idx, 'Epoch']}")
print(f"(metric_for_best_model='eval_loss', greater_is_better=False)")

 Epoch  Training Loss  Validation Loss BLEU SacreBLEU chrF
   1.0      40.263843         7.318746 None      None None
   2.0      31.434590         3.193068 None      None None
   3.0      25.267129         2.886844 None      None None
   4.0      19.521338         2.348637 None      None None
   5.0      16.019686         1.776483 None      None None
   6.0      12.886819         1.588331 None      None None
   7.0      10.741406         1.481988 None      None None
   8.0       8.827051         1.418644 None      None None
   9.0       7.334408         1.381941 None      None None
  10.0       6.176025         1.369326 None      None None

Best epoch by Validation Loss: epoch 10.0
(metric_for_best_model='eval_loss', greater_is_better=False)


## Step 15: Sample Translations

5–10 example comparisons for **Report Section 5**.

In [13]:
import random
random.seed(42)

sample_idx = random.sample(range(len(test_dataset)), 10)

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

rows = []
for i in sample_idx:
    example = test_dataset[i]
    inputs = tokenizer(example["source_text"], return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH).to(device)
    with torch.no_grad():
        generated = model.generate(
    **inputs,
    max_length=MAX_TARGET_LENGTH,
    num_beams=4,
    no_repeat_ngram_size=3,
    repetition_penalty=1.3,
)
    prediction = tokenizer.decode(generated[0], skip_special_tokens=True)

    rows.append({
        "Source": example["source_text"],
        "Reference": example["target_text"],
        "Prediction": prediction,
        "Direction": example["translation_direction"],
    })

samples_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 100)
samples_df


,Source,Reference,Prediction,Direction
0,"translate English to Dholuo: In the meantime, KRA urges all the patriotic citizens of Kenya to s...","E kindeno bende, KRA jiwo raia duto mag Kenya mondo ochung ' kor sirkal, to moloyo e kinde matekni.",Jodak nyalo yudo e bwo chenro mar Kenya.,eng-luo
1,tafsiri Kiswahili kwa Dholuo: Kikao cha robo hii kinaangazia usajili wa Mamlaka ya Afya ya Jamii...,"Oboke ma ng'ato nyalo ng'eyo e dwe mar ang'wen, ogolo ler kuom ng'ado bura mar ng'ado bura mar r...","Jodak nyalo yudo e klinik mag sirkal, kod ng'ado bura mar thieth mag piny (SHA) ma onge chudo ku...",swh-luo
2,tafsiri Kiswahili kwa Dholuo: Taarifa ya umma ya mwezi huu inahusisha mradi wa NARIGP unaosaidia...,Oboke mar dwe ni mar ji duto oting'o chenro mar dongruok mar piny mangima mar pur kod gwenge (NA...,"Jodak nyalo yudo kony mag piny, kod ng'iyo e bwo chenro mar ng'ado bura, kod chiwo chanjo ma ng'...",swh-luo
3,tafsiri Kiswahili kwa Dholuo: Kikao cha robo hii kinaangazia mabaraza ya ushiriki wa umma katika...,Oboke mar dwe mar ang'wen no nyiso kaka oganda nyalo bedo gi ng'iyo e chenro mag loso bajet mar ...,Jodak nyalo yudo kony e bwo chenro mar ng'ado bura mag sirkal.,swh-luo
4,translate English to Dholuo: Understanding the Huduma Namba registration system helps citizens e...,Ng'eyo chenro mar ndiko nying' jo Huduma Namba konyo jodak e loso maber ahinya gi sirikal ma gwe...,Jodak nyalo yudo e bwo chenro mar Huduma Namba.,eng-luo
5,tafsiri Kiswahili kwa Dholuo: Ofisi za eneo zinaratibu juhudi kuhusu usajili wa wakulima kwenye ...,Opis mag gwenge ochung' ne tich mar ndiko nying' jopur e sistem mar ng'eyo weche mag pur mag Ken...,"Jodak nyalo yudo kony mag piny mangima, kod ng'iyo e bwo chenro mar lando weche mag NCPB, kod ch...",swh-luo
6,tafsiri Kiswahili kwa Dholuo: Kuelewa ushiriki wa umma katika utungaji sheria hurahisisha kuripo...,Ng'eyo kaka oganda donjo e keto chike miyo bedo mayot mondo ior ripot mar tim marach ma itimo e ...,Jodak nyalo yudo e bwo chenro mar ng'ado bura.,swh-luo
7,translate English to Dholuo: Community members are advised to check on county government notices...,Jokanyo mag gwenge omiye rieko mondo otim nonro kuom lendo mag sirkal mar kaunti kuom chiwo tich...,Jodak nyalo yudo e bwo chenro mar ng'ado bura.,eng-luo
8,translate English to Dholuo: Local offices are coordinating efforts on Higher Education Loans Bo...,"Opis mag gwenge loso tich matek e wi pesa mag somo ma malo (HELB) kod kwayo mag pesa mag somo, c...","Jodak nyalo yudo kony mag sirkal, kod chenro mar lando weche mag piny (HELB) ma onge chudo e sku...",eng-luo
9,tafsiri Kiswahili kwa Dholuo: Mkakati wa wiki hii unahusisha mpito kwenda shule ya upili ya chin...,Oboke ma jumani kae biro wuoyo kuom loko ng'ato mondo odhi e skul ma piny (JSS) e bwo chenro mar...,Jodak nyalo yudo e bwo chenro mar piny (HELB).,swh-luo



## Step 16: Final Test-Set Evaluation

In [15]:
import gc

torch.cuda.empty_cache()
gc.collect()

model.generation_config.no_repeat_ngram_size = 3
model.generation_config.repetition_penalty = 1.3

trainer.compute_metrics = compute_metrics
trainer.args.predict_with_generate = True
trainer.args.generation_num_beams = 4
trainer.args.per_device_eval_batch_size = 4      # <-- shrink from 32
trainer.args.eval_accumulation_steps = 10        # <-- move logits off GPU periodically

test_results = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")
print("Test set results:")
for k, v in test_results.items():
    print(f"  {k}: {v}")

Training Loss,Validation Loss,Epoch,Bleu,Sacrebleu,Chrf
No log,1.739590,0,0.036511,3.651085,19.518976


Test set results:
  test_loss: 1.7395902872085571
  test_bleu: 0.036510845354596935
  test_sacrebleu: 3.6510845354596912
  test_chrf: 19.51897566501971


## Step 17: Save Final Model

Saved to Google Drive (per the mandatory Drive-mount requirement), plus a
zipped copy for submission.

In [16]:
trainer.save_model(MODEL_OUT_DIR)
tokenizer.save_pretrained(MODEL_OUT_DIR)
print(f"Model saved to {MODEL_OUT_DIR}")

import shutil
zip_path = f"{DRIVE_PROJECT_DIR}/models/mt5-dholuo-final"
shutil.make_archive(zip_path, "zip", MODEL_OUT_DIR)
print(f"Zipped model available at {zip_path}.zip")


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.32s/it]


Model saved to /home/jovyan/mt5-project/models/mt5-dholuo-final
Zipped model available at /home/jovyan/mt5-project/models/mt5-dholuo-final.zip


---
## Report Scaffold (fill in after training completes)

Use this section as the basis for `Trizah — mT5 Fine-Tuned for Dholuo` (the
Week 3 submission report).

### 1. Setup
- **Base model:** `google/mt5-small`
- **Why selected:** assigned model for this track; also a natural comparison
  point against Steve's NLLB-Dholuo run since both target the same language.
- **Language support:** Dholuo is **not natively supported** by mT5 (not in
  its mC4 pretraining data). mT5 has no per-language embedding table to
  extend, so the workaround used here is a **consistent text-prefix scheme**
  (`"translate English to Dholuo: "` / `"tafsiri Kiswahili kwa Dholuo: "`)
  rather than an embedding modification — see the intro cell for the full
  rationale.

### 2. Dataset
- Source: `data/processed/psa_dataset_dholuo_somali_cleaned.csv` (16,029 base
  rows → run Step 6 for the exact post-dedup pair count).
- Split: 80/10/10, stratified by source language — see Step 7 output for exact
  counts.
- Preprocessing carried over from `dholuo_somali_preprocessing_eda.ipynb`:
  curly-quote normalization, embedded-caption stripping, mojibake correction.
  Known limitation carried over: Dholuo translation quality has no automated
  validation tool (no `langdetect`/fastText coverage) — depends on Rencia's
  native-speaker QA pass.

### 3. Results
*(paste the Step 14 table here once training finishes)*
- Best epoch: *(from Step 14 output)*
- Selection metric: SacreBLEU (`metric_for_best_model="sacrebleu"`), chosen
  over raw BLEU because SacreBLEU is corpus-level, tokenization-standardized,
  and directly comparable across the team's different models.

### 5. Sample Translations
*(paste the Step 15 table here)*

### 6. Challenges Faced
- Dataset size: ~32k translation pairs (16k rows × 2 source languages) means
  10 full epochs on mT5-small is a meaningful Colab time commitment — note
  actual wall-clock time from Step 13 here.
- mT5 has no native Dholuo support, so early-epoch outputs may be close to
  copying the source or degenerate; note whether this resolved by later
  epochs.
- *(add anything Colab-specific: disconnects, OOM, etc.)*

### 7. Limitations
- A large share of the underlying English/Kiswahili source sentences are
  synthetically generated (fact-grounded, not organic PSA text) — noted
  transparently in Week 1; this may affect how naturally the model's Dholuo
  output reads on real-world PSA phrasing.
- No automated Dholuo QA tool exists, so reported translation quality is
  bounded by manual review coverage.
- Potential improvements: more epochs, larger mT5 checkpoint (base vs small),
  incorporating the glossary (`psa_glossary_dholuo_somali.csv`) to enforce
  consistent handling of institution names/acronyms during generation.
